# 16 · Spark joins in depth (Spark)

How Spark runs a join:
- **Broadcast hash join**: the small side is copied to every executor. No shuffle. Fastest.
- **Sort-merge join**: both sides are **shuffled** (moved over the network) by the key, sorted, then merged.
  The default for two big tables.
- **Skew**: one key has far more rows than the others, so one task does most of the work and the job waits for it.

`how=` values: `inner`, `left`, `right`, `full`, `left_semi`, `left_anti`, `cross`.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/16_joins_deep_dive.ipynb`](../../spark/practice/16_joins_deep_dive.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. All join types in one table  ·  🟢 easy

Join `customers` to the **distinct customer_ids** of `orders`. Print the row count for each `how`: inner, left, right, full, left_semi, left_anti.

**How it works:** Run the same join with each `how` and count: shows which rows each join type keeps.

In [ ]:
buyers = orders.select("customer_id").distinct()
for how in ["inner", "left", "right", "full", "left_semi", "left_anti"]:
    print(f"{how:10}", customers.join(buyers, "customer_id", how).count())

inner      32
left       40
right      32
full       40
left_semi  32
left_anti  8

**Why / traps:** 40 customers, 32 buyers. semi = inner without the right columns (and no duplicates); anti = left rows **without** a match (8 never ordered).

## Q2. Ambiguous column names  ·  🟡 medium

Join employees to departments with an **expression** (`e.dept_id == d.dept_id`) and select `dept_id`. It fails: why? Fix it two ways: (1) join on the column **name**, (2) use aliases.

**How it works:** After an expression join both tables' `dept_id` exist, so a plain name is ambiguous. Joining on the name, or selecting `e.dept_id`, fixes it.

In [ ]:
e, d = employees.alias("e"), departments.alias("d")
joined = e.join(d, F.col("e.dept_id") == F.col("d.dept_id"))
try:
    joined.select("dept_id").show(1)
except Exception as err:
    print("error:", type(err).__name__)

employees.join(departments, "dept_id").select("dept_id", "name", "dept_name").show(2)   # fix 1
joined.select(F.col("e.dept_id"), "e.name", "d.dept_name").show(2)                     # fix 2

error: AnalysisException
+-------+-------------+-----------+
|dept_id|         name|  dept_name|
+-------+-------------+-----------+
|      1|Alice Johnson|Engineering|
|      1|    Bob Smith|Engineering|
+-------+-------------+-----------+
only showing top 2 rows
+-------+-------------+-----------+
|dept_id|         name|  dept_name|
+-------+-------------+-----------+
|      1|Alice Johnson|Engineering|
|      1|    Bob Smith|Engineering|
+-------+-------------+-----------+
only showing top 2 rows

**Why / traps:** With an expression both `dept_id` columns stay, so the name is **ambiguous**. `on="dept_id"` (or a list of names) keeps only one copy.

## Q3. NULL keys never match  ·  🟡 medium

Make two small DataFrames whose key column has a NULL. Join them with `==` and with `eqNullSafe`. Compare the results.

*Expected result: 2 rows · columns: `va`, `vb`*

**How it works:** `==` never matches NULL keys; `eqNullSafe` treats NULL = NULL as true.

In [ ]:
a = spark.createDataFrame([(1, "a1"), (None, "a-null")], "k int, va string")
b = spark.createDataFrame([(1, "b1"), (None, "b-null")], "k int, vb string")
print("== join rows:", a.join(b, a.k == b.k).count())
res = a.join(b, a.k.eqNullSafe(b.k)).select("va", "vb")
res.show()

== join rows: 1

+------+------+
|    va|    vb|
+------+------+
|    a1|    b1|
|a-null|b-null|
+------+------+

**Why / traps:** `NULL = NULL` is unknown, so NULL keys never join. Usually that is right. If NULL is a real value for you ("no department"), use `eqNullSafe` (SQL `<=>`) or COALESCE the key.

## Q4. Join on two columns with different names  ·  🟢 easy

Join `order_items` (order_id, product_id) to a small price list `prices(oid, pid, unit_price)` on both keys. Show matching rows.

*Expected result: 2 rows · columns: `order_id`, `product_id`, `quantity`, `unit_price`*

**How it works:** Build the join condition with `&` between two equalities when the column names differ.

In [ ]:
%%sql
SELECT i.order_id, i.product_id, i.quantity, p.unit_price
FROM order_items i
JOIN VALUES (1001, 7, 30.0), (1001, 10, 30.0), (1002, 99, 1.0) AS p(oid, pid, unit_price)
  ON i.order_id = p.oid AND i.product_id = p.pid

,order_id,product_id,quantity,unit_price
0,1001,7,1,30.0
1,1001,10,2,30.0


**DataFrame API:**

In [ ]:
prices = spark.createDataFrame([(1001, 7, 30.0), (1001, 10, 30.0), (1002, 99, 1.0)],
                               "oid int, pid int, unit_price double")
res = (order_items.join(prices, (order_items.order_id == prices.oid) & (order_items.product_id == prices.pid))
       .select("order_id", "product_id", "quantity", "unit_price"))
res.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|    1001|         7|       1|      30.0|
|    1001|        10|       2|      30.0|
+--------+----------+--------+----------+

## Q5. Broadcast or not? Read the plan  ·  🟡 medium

Join `order_items` to `products` (1) with AQE broadcast **turned off** (`spark.sql.autoBroadcastJoinThreshold = -1`) and (2) with `F.broadcast`. Print the physical plan of both and find `SortMergeJoin` vs `BroadcastHashJoin`.

**How it works:** Turn off auto-broadcast to see the default SortMergeJoin with shuffles; `F.broadcast` changes it to BroadcastHashJoin.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")      # to see the plain plan
order_items.join(products, "product_id").explain()
order_items.join(F.broadcast(products), "product_id").explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
spark.conf.set("spark.sql.adaptive.enabled", "true")

== Physical Plan ==
*(5) Project [product_id#22, order_id#21, quantity#23, product_name#18, category#19, price#20]
+- *(5) SortMergeJoin [product_id#22], [product_id#17], Inner
   :- *(2) Sort [product_id#22 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(product_id#22, 4), ENSURE_REQUIREMENTS, [plan_id=8541]
   :     +- *(1) Filter isnotnull(product_id#22)
   :        +- FileScan csv [order_id#21,product_id#22,quantity#23] Batched: false, DataFilters: [isnotnull(product_id#22)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:<repo>/data/order_items.tsv], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:int,product_id:int,quantity:int>
   +- *(4) Sort [product_id#17 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(product_id#17, 4), ENSURE_REQUIREMENTS, [plan_id=8549]
         +- *(3) Filter isnotnull(product_id#17)
            +- FileScan csv [product_id#17,product_name#18,category#19,price#20] Batched: false,

**Why / traps:** SortMergeJoin needs an `Exchange hashpartitioning` (shuffle) on **both** sides. Broadcast needs `BroadcastExchange` of the small side only. Only broadcast **small** tables: it must fit in each executor's memory (and the driver's).

## Q6. Find skewed keys  ·  🟡 medium

Make a fake big table: 1,000,000 rows where key `0` has 90% of the rows. Count rows per key and show the top 3 keys.

**How it works:** groupBy(key).count() sorted by count shows the hot key at the top.

In [ ]:
big = spark.range(1_000_000).withColumn(
    "key", F.when(F.col("id") % 10 != 0, F.lit(0)).otherwise(F.col("id") % 100))
big.groupBy("key").count().orderBy(F.desc("count")).show(3)

+---+------+
|key| count|
+---+------+
|  0|910000|
| 30| 10000|
| 10| 10000|
+---+------+
only showing top 3 rows

**Why / traps:** Always check the key distribution before a big join or groupBy. The task with key 0 would get ~900,000 rows while the others get ~1,000.

## Q7. Fix skew with salting  ·  🔴 hard

Join the skewed `big` table to a small `dim(key, label)` with **salting**: add a random salt 0–9 to `big`, copy each `dim` row 10 times (one per salt), join on (key, salt). Check the result count is still 1,000,000.

**How it works:** Add a random salt (0–9) on the big side, copy the small side once per salt, and join on (key, salt): the hot key is spread over 10 tasks.

In [ ]:
SALT = 10
big = spark.range(1_000_000).withColumn(
    "key", F.when(F.col("id") % 10 != 0, F.lit(0)).otherwise(F.col("id") % 100))
dim = spark.range(100).select(F.col("id").alias("key"), F.concat(F.lit("k"), "id").alias("label"))

big_salted = big.withColumn("salt", (F.rand(seed=42) * SALT).cast("int"))
dim_salted = dim.crossJoin(spark.range(SALT).withColumnRenamed("id", "salt"))
joined = big_salted.join(dim_salted, ["key", "salt"])
print(joined.count())
joined.groupBy("salt").count().orderBy("salt").show(3)

1000000
+----+------+
|salt| count|
+----+------+
|   0| 99990|
|   1|100311|
|   2| 99927|
+----+------+
only showing top 3 rows

**Why / traps:** The hot key 0 is now split into 10 smaller groups, so 10 tasks share it. Cost: the small side is 10× bigger. **First try** a broadcast join (no shuffle = no skew) or AQE skew join (`spark.sql.adaptive.skewJoin.enabled`, on by default in Spark 3.2+).

## Q8. Range join (non-equi)  ·  🟡 medium

Give each employee a salary band from a band table: `low` (0–79,999), `mid` (80,000–129,999), `high` (130,000+). Join on `salary BETWEEN min AND max`.

*Expected result: 30 rows · columns: `name`, `salary`, `band`*

**How it works:** A join condition with BETWEEN (no equality) matches each employee to the band whose range contains the salary.

In [ ]:
%%sql
SELECT e.name, e.salary, b.band
FROM employees e
JOIN VALUES ('low', 0, 79999), ('mid', 80000, 129999), ('high', 130000, 1000000000) AS b(band, min_sal, max_sal)
  ON e.salary BETWEEN b.min_sal AND b.max_sal

,name,salary,band
0,Alice Johnson,250000.00,high
1,Bob Smith,150000.00,high
2,Carol White,120000.00,mid
3,David Brown,120000.00,mid
4,Eva Green,110000.00,mid
5,Frank Moore,95000.00,mid
6,Grace Lee,95000.00,mid
7,Henry Clark,140000.00,high
8,Irene Lewis,90000.00,mid
9,Jack Walker,90000.00,mid


**DataFrame API:**

In [ ]:
bands = spark.createDataFrame([("low", 0, 79999), ("mid", 80000, 129999), ("high", 130000, 10**9)],
                              "band string, min_sal int, max_sal int")
res = (employees.join(F.broadcast(bands),
                      employees.salary.between(bands.min_sal, bands.max_sal))
       .select("name", "salary", "band"))
res.show()

+-------------+---------+----+
|         name|   salary|band|
+-------------+---------+----+
|Alice Johnson|250000.00|high|
|    Bob Smith|150000.00|high|
|  Carol White|120000.00| mid|
|  David Brown|120000.00| mid|
|    Eva Green|110000.00| mid|
|  Frank Moore| 95000.00| mid|
|    Grace Lee| 95000.00| mid|
|  Henry Clark|140000.00|high|
|  Irene Lewis| 90000.00| mid|
|  Jack Walker| 90000.00| mid|
|   Karen Hall| 90000.00| mid|
|    Leo Allen| 70000.00| low|
|    Mia Young| 65000.00| low|
|  Nathan King|130000.00|high|
| Olivia Scott| 85000.00| mid|
|   Paul Adams| 85000.00| mid|
|  Quinn Baker| 60000.00| low|
|Rachel Nelson|135000.00|high|
|   Sam Carter|100000.00| mid|
|Tina Mitchell| 98000.00| mid|
+-------------+---------+----+
only showing top 20 rows

**Why / traps:** A join without `=` cannot be hash/sort-merged: Spark uses a **nested loop**. Broadcast the small table, or it gets very slow.

## Q9. Cross join explosion  ·  🟢 easy

How many rows do `customers × products` make? And `orders × order_items` (without a condition)? Print both counts.

**How it works:** `crossJoin` pairs every row with every row: count = rows × rows.

In [ ]:
print(customers.crossJoin(products).count())
print(orders.crossJoin(order_items).count())

800
21708

**Why / traps:** Rows multiply: 40 × 20 = 800, 108 × 201 = 21,708. A missing join condition on big tables can create billions of rows. Spark makes you write `crossJoin` on purpose.